<a href="https://colab.research.google.com/github/Jirakit2533/GitHub/blob/main/Number_Extracter_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1: ติดตั้ง EasyOCR และเครื่องมือที่เกี่ยวข้อง
# ==========================================
!pip install easyocr opencv-python-headless matplotlib -q

import cv2
import easyocr
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("🔄 กำลังโหลดโมเดล EasyOCR... (รอบแรกอาจใช้เวลาดาวน์โหลดสิบล้านพิกเซลสักครู่)")
# โหลดเฉพาะโมเดลภาษาอังกฤษ 'en' สำหรับอ่านตัวเลขดิจิทัล
reader = easyocr.Reader(['en'], gpu=True)
print("✅ โหลด EasyOCR สำเร็จ! พร้อมใช้งานแบบไร้บั๊กเวอร์ชันแล้วครับ")

🔄 กำลังโหลดโมเดล EasyOCR... (รอบแรกอาจใช้เวลาดาวน์โหลดสิบล้านพิกเซลสักครู่)
✅ โหลด EasyOCR สำเร็จ! พร้อมใช้งานแบบไร้บั๊กเวอร์ชันแล้วครับ


In [ ]:
# ==========================================
# CELL 2: เวอร์ชันรองรับภาพขนาดเล็ก/ถ่ายไกล + จำกัดไม่เกิน 6 ตัว
# ==========================================
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    img = cv2.imread(file_name)

    # ----------------------------------------------------
    #  UPSCALING PROCESS (แก้ปัญหาภาพเล็ก/ถ่ายไกล)
    # ----------------------------------------------------
    # ขยายภาพขึ้น 3 เท่า (300%) โดยใช้ INTER_CUBIC เพื่อเพิ่มพิกเซลให้ตัวเลข
    scale_factor = 3
    width = int(img.shape[1] * scale_factor)
    height = int(img.shape[0] * scale_factor)
    img_large = cv2.resize(img, (width, height), interpolation=cv2.INTER_CUBIC)

    # ----------------------------------------------------
    # DIGITAL DISPLAY OPTIMIZATION (ทำบนภาพที่ขยายแล้ว)
    # ----------------------------------------------------
    gray = cv2.cvtColor(img_large, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # ปรับ Kernel เป็น (5, 5) เพื่อให้สอดคล้องกับภาพที่ขยายใหญ่ขึ้น
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    processed_img = cv2.dilate(thresh, kernel, iterations=1)

    # ----------------------------------------------------
    # SEND TO AI WITH ALLOWLIST
    # ----------------------------------------------------
    results = reader.readtext(processed_img, allowlist='0123456789.:-')

    # แผนสำรอง: หากภาพแต่งอ่านไม่ออก ให้ใช้ภาพดิบที่ขยายใหญ่แล้วช่วยอ่าน
    if len(results) == 0:
        results = reader.readtext(img_large, allowlist='0123456789.:-')

    # สกัดผลลัพธ์เป็น รายการข้อความ
    string_pieces = []
    for res in results:
        text = res[1]
        if text:
            string_pieces.append(text)

    # รวมข้อความทั้งหมด และตัดเอาเฉพาะ 6 ตัวแรก
    final_string = "".join(string_pieces)[:6]

    # ----------------------------------------------------
    # SHOW RESULTS
    # ----------------------------------------------------
    print("\n" + "="*40)
    print(f" 🎯 DIGITAL DISPLAY OUTPUT (Far-Distance): '{final_string}'")
    print("="*40 + "\n")

else:
    print("❌ ไม่พบไฟล์")

Saving S__24920170.jpg to S__24920170 (15).jpg

 🎯 DIGITAL DISPLAY OUTPUT (Far-Distance): '30'



In [ ]:
# ==========================================
# CELL 1: ติดตั้งคลังโปรแกรมของ Hugging Face
# ==========================================
!pip install transformers torch pillow -q

import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from google.colab import files

print("🔄 กำลังดาวน์โหลดโมเดล TrOCR-Printed จาก Microsoft...")
print("(ไฟล์มีขนาดประมาณ 1.3 GB รอบแรกจะใช้เวลาดาวน์โหลดประมาณ 1-2 นาทีครับ)")

# โหลด Processor (ตัวแปลงภาพ) และ Modelสำหรับตัวอักษรพิมพ์/ดิจิทัล (Printed)
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')

print("✅ โหลดโมเดล TrOCR สำเร็จ! ระบบพร้อมทำงานแล้วครับ")

🔄 กำลังดาวน์โหลดโมเดล TrOCR-Printed จาก Microsoft...
(ไฟล์มีขนาดประมาณ 1.3 GB รอบแรกจะใช้เวลาดาวน์โหลดประมาณ 1-2 นาทีครับ)


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ โหลดโมเดล TrOCR สำเร็จ! ระบบพร้อมทำงานแล้วครับ


In [ ]:
# ==========================================
# CELL 2: อัปโหลดภาพและดึงข้อความออกเป็น String
# ==========================================
print("📥 เลือกรูปภาพหน้าปัดดิจิทัลของคุณ (แนะนำภาพที่ครอปเฉพาะตัวเลข):")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    print(f"\n📢 TrOCR กำลังประมวลผลไฟล์: {file_name}")

    # 1. เปิดรูปภาพด้วย Pillow และแปลงเป็นระบบสี RGB
    image = Image.open(file_name).convert("RGB")

    # 2. แปลงรูปภาพให้อยู่ในฟอร์แมตที่ Transformer เข้าใจ (Pixel Values)
    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    # 3. สั่งให้โมเดลเจเนอเรต (ถอดรหัส) ภาพออกมาเป็นอักขระ
    print("🤖 Transformer กำลังแกะรหัสภาพ...")
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    # 4. แปลงรหัสอักขระกลับมาเป็น String ภาษาคน
    final_output_string = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # ==========================================
    # แสดงผลลัพธ์สุดท้าย
    # ==========================================
    print("\n" + "="*40)
    print(" 🎯 OUTPUT จาก TrOCR (String)")
    print("="*40)
    print(f"👉 '{final_output_string}'")
    print("="*40)
    print(f"ชนิดของข้อมูล: {type(final_output_string)}")

else:
    print("❌ ไม่พบไฟล์ที่อัปโหลด")

📥 เลือกรูปภาพหน้าปัดดิจิทัลของคุณ (แนะนำภาพที่ครอปเฉพาะตัวเลข):


Saving S__24920170.jpg to S__24920170 (2).jpg

📢 TrOCR กำลังประมวลผลไฟล์: S__24920170 (2).jpg
🤖 Transformer กำลังแกะรหัสภาพ...

 🎯 OUTPUT จาก TrOCR (String)
👉 '-'
ชนิดของข้อมูล: <class 'str'>
